# Statistical POS-Tagged Parser using the Penn Treebank

This notebook builds a statistical POS tagger and a PCFG parser from the NLTK Penn Treebank, then evaluates them on 150 sentences collected from the Daily Mirror online newspaper.

## 1. Setup and Corpus Loading

This section imports the required libraries, loads the Penn Treebank corpus, and reports basic statistics about it.

### 1.1 Import Libraries

[ Write your explanation here ]

In [13]:
import nltk
from nltk.corpus import treebank
from collections import Counter

# download the corpus the first time only
try:
    nltk.data.find("corpora/treebank")
except LookupError:
    nltk.download("treebank")

### 1.2 Load the NLTK Penn Treebank (Labeled)

Load the corpus in two forms:
- tagged word/tag pairs for the tagger.
- parsed trees for the PCFG parser.

In [14]:
tagged_sents = treebank.tagged_sents()   # for the TAGGER  (word/tag pairs)
parsed_sents = treebank.parsed_sents()   # for the PARSER  (structure trees)

print("First tagged sentence (first 6 tokens):")
print(tagged_sents[0][:6])

First tagged sentence (first 6 tokens):
[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ')]


Visualize the structure (tree form) of the first sentence

In [15]:
parsed_sents[0].pretty_print()

                                                     S                                                                         
                         ____________________________|_______________________________________________________________________   
                        |                                               VP                                                   | 
                        |                        _______________________|___                                                 |  
                      NP-SBJ                    |                           VP                                               | 
         _______________|___________________    |     ______________________|______________________________________          |  
        |          |              ADJP      |   |    |        |                PP-CLR                              |         | 
        |          |           ____|____    |   |    |        |          ________|_________          

### 1.3 Corpus Statistics

Compute basic statistics (sentences, tokens, distinct words and tags) to understand the training data. The small size (~3,900 sentences) is relevant to unknown-word errors discussed later.

In [16]:
n_sents  = len(tagged_sents)
tokens   = [tok for s in tagged_sents for tok in s]
n_tokens = len(tokens)
tagset   = {t for _, t in tokens}
vocab    = {w.lower() for w, _ in tokens}

print(f"Sentences            : {n_sents}")
print(f"Tokens (words)       : {n_tokens}")
print(f"Distinct word types  : {len(vocab)}")
print(f"Distinct POS tags    : {len(tagset)}")
print(f"Avg sentence length  : {n_tokens/n_sents:.1f} words")

Sentences            : 3914
Tokens (words)       : 100676
Distinct word types  : 11387
Distinct POS tags    : 46
Avg sentence length  : 25.7 words


## 2. Training the Statistical POS Tagger (HMM)

[ Write your explanation here ]

### 2.1 Transition Probabilities

[ Write your explanation here ]

In [17]:
from collections import defaultdict

# count how often each tag follows the previous tag
transition_counts = defaultdict(Counter)   # transition_counts[prev][curr] = count
tag_unigram = Counter()                     # how often each tag occurs overall

for sent in tagged_sents:
    prev = "<s>"                            # sentence-start marker
    for word, tag in sent:
        transition_counts[prev][tag] += 1
        tag_unigram[tag] += 1
        prev = tag

# turn counts into probabilities:  P(curr | prev) = count(prev, curr) / count(prev)
transition_prob = defaultdict(dict)
for prev, nexts in transition_counts.items():
    total = sum(nexts.values())
    for curr, c in nexts.items():
        transition_prob[prev][curr] = c / total

# sanity check: what most commonly follows a determiner (DT)?
print("Most likely tags after DT:")
for tag, p in sorted(transition_prob["DT"].items(), key=lambda x: -x[1])[:5]:
    print(f"  P({tag:4} | DT) = {p:.3f}")

Most likely tags after DT:
  P(NN   | DT) = 0.471
  P(JJ   | DT) = 0.203
  P(NNP  | DT) = 0.129
  P(NNS  | DT) = 0.077
  P(CD   | DT) = 0.023


Transition table

In [18]:
# Understanding the transition table: how grammar "flows"
print("Table size:")
print(f"  {len(transition_prob)} 'previous-tag' rows (each row is a full probability distribution)\n")

# look at several different previous-tags to see the grammar patterns
for prev in ["<s>", "DT", "JJ", "IN", "MD"]:
    top = sorted(transition_prob[prev].items(), key=lambda x: -x[1])[:3]
    shown = ", ".join(f"{tag} ({p:.2f})" for tag, p in top)
    print(f"After {prev:4} -> {shown}")

Table size:
  47 'previous-tag' rows (each row is a full probability distribution)

After <s>  -> DT (0.23), NNP (0.20), IN (0.13)
After DT   -> NN (0.47), JJ (0.20), NNP (0.13)
After JJ   -> NN (0.45), NNS (0.24), JJ (0.06)
After IN   -> DT (0.32), NNP (0.15), NN (0.11)
After MD   -> VB (0.82), RB (0.16), -NONE- (0.01)


### 2.2 Emission Probabilities

[ Write your explanation here ]

In [19]:
# count how often each tag produces each word
emission_counts = defaultdict(Counter)      # emission_counts[tag][word] = count

for sent in tagged_sents:
    for word, tag in sent:
        emission_counts[tag][word.lower()] += 1

# turn counts into probabilities:  P(word | tag) = count(tag, word) / count(tag)
emission_prob = defaultdict(dict)
for tag, words in emission_counts.items():
    total = sum(words.values())
    for word, c in words.items():
        emission_prob[tag][word] = c / total

# sanity check: the most likely words for a singular noun (NN)
print("Most likely words for NN:")
for word, p in sorted(emission_prob["NN"].items(), key=lambda x: -x[1])[:5]:
    print(f"  P({word!r:12} | NN) = {p:.4f}")

Most likely words for NN:
  P('%'          | NN) = 0.0338
  P('company'    | NN) = 0.0197
  P('year'       | NN) = 0.0163
  P('market'     | NN) = 0.0135
  P('trading'    | NN) = 0.0112


Emission table

In [20]:
# Understanding the emission table: which words belong to which tag
print("Table size:")
print(f"  {len(emission_prob)} tag rows; e.g. NN holds {len(emission_prob['NN'])} distinct words\n")

# top words for several different tags
for tag in ["NN", "VBD", "JJ", "IN", "PRP"]:
    top = sorted(emission_prob[tag].items(), key=lambda x: -x[1])[:4]
    shown = ", ".join(f"{w} ({p:.3f})" for w, p in top)
    print(f"{tag:4} -> {shown}")

Table size:
  46 tag rows; e.g. NN holds 2506 distinct words

NN   -> % (0.034), company (0.020), year (0.016), market (0.014)
VBD  -> said (0.202), was (0.121), were (0.065), had (0.053)
JJ   -> new (0.029), other (0.024), last (0.017), many (0.017)
IN   -> of (0.236), in (0.177), for (0.086), that (0.052)
PRP  -> it (0.336), he (0.177), they (0.153), i (0.066)


### 2.3 Smoothing and Unknown-Word Handling

[ Write your explanation here ]

### 2.4 Viterbi Decoding

[ Write your explanation here ]

### 2.5 Save the Tagger Model

[ Write your explanation here ]

## 3. Building the PCFG Parser

[ Write your explanation here ]

### 3.1 Extract Productions from the Parsed Trees

[ Write your explanation here ]

### 3.2 Convert Trees to Chomsky Normal Form

[ Write your explanation here ]

### 3.3 Induce the PCFG

[ Write your explanation here ]

### 3.4 Save the PCFG Parser Data

[ Write your explanation here ]

### 3.5 Parsing Sanity Check

[ Write your explanation here ]

## 4. Preparing the 150 Daily Mirror Sentences

[ Write your explanation here ]

### 4.1 Load the Raw Sentences

[ Write your explanation here ]

### 4.2 Penn Treebank Tokenization

[ Write your explanation here ]

### 4.3 Load the Gold-Standard Tags

[ Write your explanation here ]

## 5. Tagging and Parsing the 150 Sentences

[ Write your explanation here ]

### 5.1 Tag with the Trained Model

[ Write your explanation here ]

### 5.2 Parse a Sample of Sentences

[ Write your explanation here ]

### 5.3 Save the Predictions

[ Write your explanation here ]

## 6. Evaluation

[ Write your explanation here ]

### 6.1 Overall Accuracy and Baseline

[ Write your explanation here ]

### 6.2 Per-Tag Precision, Recall and F1

[ Write your explanation here ]

### 6.3 Confusion Matrix

[ Write your explanation here ]

### 6.4 Out-of-Vocabulary Analysis

[ Write your explanation here ]

### 6.5 Save the Results

[ Write your explanation here ]

## 7. Results Summary and Error Analysis

[ Write your explanation here ]